In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import time
import sys
sys.path.append('..')

from environments.line_world import LineWorldEnv
from environments.grid_world import GridWorldEnv
from environments.rock_paper_scissors import RockPaperScissorsEnv
from environments.monty_hall_1 import MontyHall1Env
from algorithms.planning import dyna_q, dyna_q_plus

print("Imports OK ✅")

Imports OK ✅


In [2]:
from environments.monty_hall_2 import MontyHall2Env

envs = {
    'LineWorld': LineWorldEnv(),
    'GridWorld': GridWorldEnv(),
    'RockPaperScissors': RockPaperScissorsEnv(),
    'MontyHall1': MontyHall1Env(),
    'MontyHall2': MontyHall2Env()   # ← ajouté !
}

print("=== Dyna-Q sur tous les environnements ===\n")

for name, env in envs.items():
    t = time.time()
    Q, pi = dyna_q(env, N=10, max_steps=10000)
    t = time.time() - t
    print(f"{name} → temps={t:.3f}s | policy={pi.argmax(axis=1)}")

=== Dyna-Q sur tous les environnements ===

LineWorld → temps=1.147s | policy=[0 1 1 1 0]
GridWorld → temps=1.535s | policy=[0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
RockPaperScissors → temps=1.355s | policy=[0 1 2 0 0]
MontyHall1 → temps=1.260s | policy=[1 0 1 1 0]
MontyHall2 → temps=1.248s | policy=[2 1 0 0 1 0 0 0 0 1 1 1 1 1 0 0 0]


In [3]:
max_steps = 5000
steps_list = list(range(1, max_steps+1))

dyna_q_rewards = []
dyna_q_plus_rewards = []

env = LineWorldEnv()
Q1, pi1 = dyna_q(env, N=10, max_steps=max_steps,
                  cumulated_rewards=dyna_q_rewards)

env = LineWorldEnv()
Q2, pi2 = dyna_q_plus(env, N=10, max_steps=max_steps,
                       kappa=0.001, cumulated_rewards=dyna_q_plus_rewards)

print("Policy Dyna-Q  :", ['Droite' if a==1 else 'Gauche' for a in pi1.argmax(axis=1)])
print("Policy Dyna-Q+ :", ['Droite' if a==1 else 'Gauche' for a in pi2.argmax(axis=1)])

# Graphique
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(dyna_q_rewards, label='Dyna-Q', color='blue', alpha=0.7)
ax.plot(dyna_q_plus_rewards, label='Dyna-Q+', color='orange', alpha=0.7)
ax.set_xlabel('Steps')
ax.set_ylabel('Reward cumulé')
ax.set_title('Dyna-Q vs Dyna-Q+ — LineWorld')
ax.legend()
plt.tight_layout()
plt.savefig('planning_lineworld.png', dpi=100, bbox_inches='tight')
plt.close()
print("Graphique sauvegardé ✅")

Policy Dyna-Q  : ['Gauche', 'Droite', 'Droite', 'Droite', 'Gauche']
Policy Dyna-Q+ : ['Droite', 'Droite', 'Droite', 'Gauche', 'Gauche']
Graphique sauvegardé ✅


In [4]:
N_values = [0, 5, 10, 50, 100]
results = {}

for N in N_values:
    rewards = []
    env = LineWorldEnv()
    dyna_q(env, N=N, max_steps=5000, cumulated_rewards=rewards)
    results[N] = rewards

fig, ax = plt.subplots(figsize=(12, 5))
for N, rewards in results.items():
    ax.plot(rewards, label=f'N={N}', alpha=0.7)

ax.set_xlabel('Steps')
ax.set_ylabel('Reward cumulé')
ax.set_title('Impact de N — Dyna-Q — LineWorld')
ax.legend()
plt.tight_layout()
plt.savefig('planning_N_impact.png', dpi=100, bbox_inches='tight')
plt.close()
print("Graphique sauvegardé ✅")

Graphique sauvegardé ✅


In [5]:
kappas = [0.0, 0.001, 0.01, 0.1, 1.0]
results_kappa = {}

for kappa in kappas:
    rewards = []
    env = LineWorldEnv()
    dyna_q_plus(env, N=10, max_steps=5000,
                kappa=kappa, cumulated_rewards=rewards)
    results_kappa[kappa] = rewards

fig, ax = plt.subplots(figsize=(12, 5))
for kappa, rewards in results_kappa.items():
    ax.plot(rewards, label=f'kappa={kappa}', alpha=0.7)

ax.set_xlabel('Steps')
ax.set_ylabel('Reward cumulé')
ax.set_title('Impact de kappa — Dyna-Q+ — LineWorld')
ax.legend()
plt.tight_layout()
plt.savefig('planning_kappa_impact.png', dpi=100, bbox_inches='tight')
plt.close()
print("Graphique sauvegardé ✅")

Graphique sauvegardé ✅


In [6]:
dyna_q_rewards_gw = []
dyna_q_plus_rewards_gw = []

env = GridWorldEnv()
Q1, pi1 = dyna_q(env, N=10, max_steps=10000,
                  cumulated_rewards=dyna_q_rewards_gw)

env = GridWorldEnv()
Q2, pi2 = dyna_q_plus(env, N=10, max_steps=10000,
                       kappa=0.001, cumulated_rewards=dyna_q_plus_rewards_gw)

# Afficher les policies
ACTION_NAMES = {0:'H', 1:'B', 2:'G', 3:'D'}
print("=== Policy Dyna-Q — GridWorld ===")
for i in range(5):
    line = ''
    for j in range(5):
        s = i*5+j
        if s == 4: line += ' [X] '
        elif s == 24: line += ' [G] '
        else: line += f'  {ACTION_NAMES[pi1.argmax(axis=1)[s]]}  '
    print(line)

print("\n=== Policy Dyna-Q+ — GridWorld ===")
for i in range(5):
    line = ''
    for j in range(5):
        s = i*5+j
        if s == 4: line += ' [X] '
        elif s == 24: line += ' [G] '
        else: line += f'  {ACTION_NAMES[pi2.argmax(axis=1)[s]]}  '
    print(line)

# Graphique rewards
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(dyna_q_rewards_gw, label='Dyna-Q', color='blue', alpha=0.7)
ax.plot(dyna_q_plus_rewards_gw, label='Dyna-Q+', color='orange', alpha=0.7)
ax.set_xlabel('Steps')
ax.set_ylabel('Reward cumulé')
ax.set_title('Dyna-Q vs Dyna-Q+ — GridWorld')
ax.legend()
plt.tight_layout()
plt.savefig('planning_gridworld.png', dpi=100, bbox_inches='tight')
plt.close()
print("Graphique sauvegardé ✅")

=== Policy Dyna-Q — GridWorld ===
  H    H    H    H   [X] 
  H    H    H    H    B  
  H    H    H    H    H  
  H    H    H    H    H  
  H    H    H    H   [G] 

=== Policy Dyna-Q+ — GridWorld ===
  B    B    D    B   [X] 
  B    B    D    D    B  
  D    B    B    B    B  
  D    D    D    D    G  
  G    H    D    D   [G] 
Graphique sauvegardé ✅
